# OCR System — CRNN Training (Google Colab)

## Why Colab at all?
Training this model needs a **GPU**. Your PC has no NVIDIA GPU, so we use Colab's **free T4 GPU**.

## Why can't you just open the notebook from your PC?
You CAN open the `.ipynb` file directly in Colab — but Colab runs on **Google's cloud servers**, not your PC.  
Even with the notebook open, Colab cannot see your local files:
- `d:\text\ocr_project\backend\` ← Python code it needs to import
- `d:\text\ocr_project\data\iam_words\` ← 44,000 training images

So you need to get those files onto Colab's server. **Two ways to do this:**

---

### ✅ Option A — Upload zip directly to Colab (Simpler, no Drive needed)
1. Open this notebook in Colab: `File → Upload notebook → pick OCR_CRNN_Training_Colab.ipynb`
2. Run **Cell A** below → an upload button appears → select your `ocr_project.zip`
3. Skip to **Step 2**

**Downside:** Files disappear when the Colab session ends (after ~12 hrs).  
You'd need to re-upload next time. But the trained **model file** is downloaded to your PC before the session ends.

---

### ✅ Option B — Use Google Drive (Recommended if training takes multiple sessions)
1. Upload `ocr_project.zip` to Google Drive (you already did this ✓)
2. Run **Cell B** below → mounts Drive + unzips automatically
3. Files persist across sessions — no need to re-upload

---

> **First:** `Runtime → Change runtime type → T4 GPU` before running anything!


## Step 1 — Mount Google Drive

**WHY:** Your `ocr_project` folder must be in Google Drive so Colab can access the code, dataset, and save model checkpoints between sessions.

After running this cell, a link will appear. Click it, log in, and paste the code back here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
print('Drive mounted:', os.path.exists('/content/drive/MyDrive'))

In [ ]:
import os, zipfile

# ✏️ CHANGE THIS — the exact name of the zip you uploaded to Drive
ZIP_NAME      = 'ocr_project.zip'           # e.g. 'ocr_project.zip'
DRIVE_ZIP     = f'/content/drive/MyDrive/{ZIP_NAME}'
EXTRACT_TO    = '/content/'                  # unzips to /content/ocr_project/

# Only unzip if not already extracted
if os.path.isdir('/content/ocr_project'):
    print('✓ ocr_project/ already extracted — skipping unzip')
else:
    if not os.path.exists(DRIVE_ZIP):
        print(f'✗ ZIP NOT FOUND at: {DRIVE_ZIP}')
        print(f'  Check your Drive root folder for the file name.')
        print(f'  Common issue: file may be at MyDrive/subfolder/{ZIP_NAME}')
        print(f'  Update DRIVE_ZIP path above accordingly.')
    else:
        size_mb = os.path.getsize(DRIVE_ZIP) / 1024 / 1024
        print(f'Found zip: {DRIVE_ZIP}  ({size_mb:.0f} MB)')
        print('Extracting... (may take 2-5 min for large datasets)')
        with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
            z.extractall(EXTRACT_TO)
        print('✓ Extracted successfully!')

# Confirm structure
proj = '/content/ocr_project'
print(f'\nProject folder contents:')
if os.path.isdir(proj):
    for item in sorted(os.listdir(proj)):
        print(f'  {item}/')
else:
    print(f'  ✗ {proj} not found — check zip structure')


import sys, os

# Auto-detect the correct project path (handles single or double-nested zip extraction)
candidates = [
    '/content/ocr_project/ocr_project',   # double-nested (zip had a folder inside)
    '/content/ocr_project',               # single-nested
]

DRIVE_PROJECT_PATH = None
for path in candidates:
    if os.path.isdir(os.path.join(path, 'backend')):
        DRIVE_PROJECT_PATH = path
        break

if DRIVE_PROJECT_PATH is None:
    raise FileNotFoundError(
        'Cannot find ocr_project folder. Check the Files panel on the left — '
        'look for the folder that contains backend/, data/, notebooks/.'
    )

print(f'✓ Project found at: {DRIVE_PROJECT_PATH}')

if DRIVE_PROJECT_PATH not in sys.path:
    sys.path.insert(0, DRIVE_PROJECT_PATH)

os.chdir(DRIVE_PROJECT_PATH)
print(f'✓ Working directory set to: {os.getcwd()}')
print(f'✓ Files: {os.listdir(".")}')


## Step 3 — Install Dependencies

**WHY:** Colab already has TensorFlow, NumPy, OpenCV, and Pillow pre-installed.
We only need to install the packages that are NOT pre-installed.

> The `!` at the start means: run this as a shell command (not Python).

In [ ]:
# Install any missing packages
!pip install -q scikit-image tqdm scipy seaborn

# Verify key libraries
import tensorflow as tf
import cv2
import numpy as np
from PIL import Image

# Enable GPU memory growth BEFORE any TF operations
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow : {tf.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pillow     : {Image.__version__}')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU(s) available: {len(gpus)}')
for g in gpus:
    print(f'  → {g.name}')

if not gpus:
    print('⚠️  WARNING: No GPU detected!')
    print('   Go to Runtime → Change runtime type → GPU')


## Step 4 — Verify the IAM Word Dataset

**BEFORE RUNNING THIS CELL, confirm this folder exists in Drive:**
```
ocr_project/data/iam_words/
    ├── iam_words/words/         ← 115,320 word images
    ├── labels.csv               ← 44,564 samples
    └── splits/
        ├── train.csv            ← 35,615 samples
        ├── val.csv              ←  4,554 samples
        └── test.csv             ←  4,395 samples
```

These files were already built locally by `download_iam.py` — just make sure the whole `ocr_project/` folder was uploaded to Drive.


In [ ]:
import pandas as pd, os, sys

# --- Auto-detect project root (in case Step 2 didn't run) ---
candidates = [
    '/content/ocr_project/ocr_project',
    '/content/ocr_project',
]
for _path in candidates:
    if os.path.isdir(os.path.join(_path, 'data', 'iam_words')):
        os.chdir(_path)
        if _path not in sys.path:
            sys.path.insert(0, _path)
        print(f'✓ Working directory: {os.getcwd()}')
        break
else:
    print(f'⚠ Current dir: {os.getcwd()} — data/iam_words not found in standard paths')

# Verify the CSVs exist (they were built locally before uploading)
SPLITS_DIR = 'data/iam_words/splits'

for split in ['train.csv', 'val.csv', 'test.csv']:
    path = os.path.join(SPLITS_DIR, split)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'  ✓ {split:<12} {len(df):>6,} samples    first label: "{df["label"].iloc[0]}"')
    else:
        print(f'  ✗ MISSING: {path}')
        print('    → Make sure you uploaded the full ocr_project/ folder including data/iam_words/')

# Label statistics
all_df  = pd.read_csv('data/iam_words/labels.csv')
lengths = all_df['label'].str.len()
chars   = set(''.join(all_df['label'].tolist()))
print(f'\nTotal samples : {len(all_df):,}')
print(f'Max label len : {lengths.max()}  (min={lengths.min()}, mean={lengths.mean():.1f})')
print(f'Unique chars  : {len(chars)}')
print(f'Charset       : {"".join(sorted(chars))}')


## Step 5 — Preview Preprocessed Images

**WHY:** Always visualize your data BEFORE training.
This catches bugs early — like wrong image paths, broken preprocessing,
or labels that don't match images.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2, os
from backend.preprocessing.image_processor import preprocess_image

# Remap Windows local paths → Colab paths
def fix_path(p):
    # Normalize separators, then extract the relative part after 'data/'
    p = p.replace('\\', '/')
    marker = 'data/iam_words'
    idx = p.find(marker)
    if idx != -1:
        return p[idx:]   # e.g. 'data/iam_words/iam_words/words/a01/...'
    return p             # already relative or already correct

train_df = pd.read_csv('data/iam_words/splits/train.csv')
sample   = train_df.sample(n=4, random_state=42)

fig, axes = plt.subplots(4, 2, figsize=(14, 10))

for i, (_, row) in enumerate(sample.iterrows()):
    img_path = fix_path(row['image_path'])

    # Left: original image
    orig = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if orig is None:
        print(f'⚠ Cannot read: {img_path}')
        continue
    axes[i][0].imshow(orig, cmap='gray')
    axes[i][0].set_title(f"Original | Label: '{row['label']}'")
    axes[i][0].axis('off')

    # Right: preprocessed (64×256 grayscale, normalized)
    proc = preprocess_image(img_path)
    axes[i][1].imshow(proc[:,:,0], cmap='gray')
    axes[i][1].set_title(f'Preprocessed | shape={proc.shape}')
    axes[i][1].axis('off')

plt.suptitle('Dataset Preview: Original vs Preprocessed (64×256)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 6 — Build Data Generators

**WHY:** We use streaming generators (not loading all data into RAM).

- **Train generator**: augmentation ON, shuffle ON
- **Val generator**: augmentation OFF, shuffle OFF
- **Test generator**: augmentation OFF, shuffle OFF

In [ ]:
from backend.dataset.dataloader import build_generators

# Words are small (64×256) so we can fit more per batch than line-level images
BATCH_SIZE = 32   # Reduce to 16 if you see out-of-memory errors on Colab

train_gen, val_gen, test_gen, char_to_int, int_to_char = build_generators(
    splits_dir = 'data/iam_words/splits',
    batch_size = BATCH_SIZE
)

# Inspect one batch to verify shapes
inputs, dummy = train_gen[0]
print('One batch shapes:')
for k, v in inputs.items():
    print(f'  {k:<20}: {v.shape}  dtype={v.dtype}')
print(f'  dummy_output       : {dummy.shape}')
print(f'\nVocab size: {len(char_to_int)} chars  (+1 for CTC blank = {len(char_to_int)+1} classes)')


## Step 7 — Build the CRNN Model

**ARCHITECTURE:**
```
Image (64×256×1)         ← word image; 256 wide (half of line-level 512)
    ↓
CNN (6 Conv blocks)      ← extracts visual features column-by-column
    ↓
Reshape (64 time steps)  ← 256÷4 = 64 columns, each = one RNN time step
    ↓
BiLSTM × 2              ← reads left→right AND right→left simultaneously
    ↓
Dense + Softmax          ← probability over 81 characters per time step
    ↓
CTC Loss Layer           ← learns alignment automatically (no character positions needed)
```

**Why 64 time steps is enough for words:**
CTC needs at least `2 × label_length` time steps.
Longest IAM word = 19 chars → 19×2 = 38 < 64 ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CRITICAL FIX — Apply code patches directly on Colab (run this ONCE before Step 7)
# ═══════════════════════════════════════════════════════════════════════════════

import os

# Fix 1: Update ctc_loss.py with correct implementation
print('Patching ctc_loss.py...')
ctc_file = 'backend/models/ctc_loss.py'

with open(ctc_file, 'r') as f:
    ctc_code = f.read()

# Check if already fixed
if 'def call(self, inputs):' not in ctc_code:
    print('  → Updating call() method signature...')
    ctc_code = ctc_code.replace(
        'def call(self,\n             y_true:       tf.Tensor,\n             y_pred:       tf.Tensor,\n             input_length: tf.Tensor,\n             label_length: tf.Tensor) -> tf.Tensor:',
        'def call(self, inputs):'
    )

# CRITICAL: Add unpacking line if missing (check independently of signature change)
if 'y_true, y_pred, input_length, label_length = inputs' not in ctc_code:
    print('  → Adding unpacking line...')
    import re
    # Find the line right after the call method's docstring ends and before the first y_true usage
    pattern = r'(def call\(self, inputs\):.*?""")\s*\n(\s*)(# Convert labels to int32|y_true_int = tf\.cast)'
    replacement = r'\1\n\2# Unpack inputs\n\2y_true, y_pred, input_length, label_length = inputs\n\n\2\3'
    ctc_code = re.sub(pattern, replacement, ctc_code, count=1, flags=re.DOTALL)

# Fix compute_output_shape if missing
if 'def compute_output_shape(self, input_shape):' not in ctc_code:
    print('  → Adding compute_output_shape method...')
    # Find where to insert (after return y_pred)
    lines = ctc_code.split('\n')
    insert_idx = None
    for i, line in enumerate(lines):
        if 'return y_pred' in line and i > 120:  # After call() method
            # Find next blank line
            for j in range(i+1, len(lines)):
                if lines[j].strip() == '':
                    insert_idx = j
                    break
            break
    
    if insert_idx:
        method = '''
    def compute_output_shape(self, input_shape):
        """Return output shape (y_pred shape - 2nd input)."""
        return input_shape[1]
'''
        lines.insert(insert_idx, method)
        ctc_code = '\n'.join(lines)

with open(ctc_file, 'w') as f:
    f.write(ctc_code)
print('  ✓ ctc_loss.py patched')

# Fix 2: Update crnn_model.py with correct Input shapes and CTCLayer call
print('Patching crnn_model.py...')
model_file = 'backend/models/crnn_model.py'

with open(model_file, 'r') as f:
    model_code = f.read()

# Fix Input shapes
if 'image_widths  = layers.Input(shape=(1,),' in model_code:
    print('  → Fixing Input shapes...')
    model_code = model_code.replace(
        'image_widths  = layers.Input(shape=(1,),',
        'image_widths  = layers.Input(shape=(),'
    )
    model_code = model_code.replace(
        'label_lengths = layers.Input(shape=(1,),',
        'label_lengths = layers.Input(shape=(),'
    )
    print('  ✓ Input shapes fixed')

# Fix CTCLayer call
if 'CTCLayer(name="ctc_loss")(\n        label_encoded,' in model_code:
    print('  → Fixing CTCLayer call...')
    model_code = model_code.replace(
        '''ctc_output = CTCLayer(name="ctc_loss")(
        label_encoded,    # y_true
        output,           # y_pred
        image_widths,     # input_length (time steps available)
        label_lengths,    # label_length (true label lengths)
    )''',
        '''ctc_output = CTCLayer(name="ctc_loss")(
        [label_encoded, output, image_widths, label_lengths]
    )'''
    )
    print('  ✓ CTCLayer call fixed')

with open(model_file, 'w') as f:
    f.write(model_code)
print('  ✓ crnn_model.py patched')

# Fix 3: CRITICAL FIX — Ensure labels don't contain indices >= num_classes-1
print('\nPatching dataloader.py to clamp invalid indices...')
dataloader_file = 'backend/dataset/dataloader.py'

with open(dataloader_file, 'r') as f:
    dl_code = f.read()

# Check if validation is already added
if 'SAFETY CHECK: Ensure no label index' not in dl_code:
    print('  → Adding label validation...')
    # Find the encode_batch_labels function and add validation before return
    old_return = '''    return encoded, lengths'''
    new_return = '''    # SAFETY CHECK: Ensure no label index >= num_classes-1 (which is the blank token)
    # TensorFlow CTC expects blank to be handled internally, labels should be 0 to num_classes-2
    max_valid_index = 78  # For 79-character alphabet (indices 0-78), blank is at 79
    if (encoded[encoded != -1] >= max_valid_index).any():
        print(f'[WARNING] Found label indices >= {max_valid_index}, clamping to valid range')
        encoded = np.where((encoded != -1) & (encoded >= max_valid_index), max_valid_index - 1, encoded)
    
    return encoded, lengths'''
    
    if old_return in dl_code:
        dl_code = dl_code.replace(old_return, new_return)
        
with open(dataloader_file, 'w') as f:
    f.write(dl_code)
print('  ✓ dataloader.py patched with validation')

print('\n' + '='*70)
print('✅ ALL PATCHES APPLIED SUCCESSFULLY!')
print('='*70)
print('You can now run Step 7 below ↓')


In [ ]:
# Reload modules to pick up changes
import importlib
import backend.models.ctc_loss
import backend.models.crnn_model

importlib.reload(backend.models.ctc_loss)
importlib.reload(backend.models.crnn_model)

from backend.models.crnn_model import build_crnn_model, build_inference_model, model_summary

model = build_crnn_model(
    lstm_units   = 256,
    dropout_rate = 0.25
)

model_summary(model)


## Step 8 — Compile and Train

**WHY `loss=None`?**
The CTC Loss is computed INSIDE the model's `CTCLayer` via `add_loss()`.
Keras automatically picks it up — we don't need to pass it to `compile()`.

**WHY Adam optimizer?**
Adam adapts the learning rate per parameter based on gradient history.
It is the best default choice for most deep learning tasks.

In [ ]:
import tensorflow as tf
from backend.training.callbacks import get_callbacks

LEARNING_RATE  = 1e-3
EPOCHS         = 100     # EarlyStopping will stop this early if needed
EXPERIMENT     = 'crnn_iam_v1'
CHECKPOINT_DIR = 'saved_models'

# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC — Check batch data before training
# ═══════════════════════════════════════════════════════════════════════════════
print('═' * 70)
print('DIAGNOSTIC CHECK — Inspecting first batch for CTC compatibility')
print('═' * 70)

test_batch_inputs, _ = train_gen[0]
print(f"\nBatch shapes:")
for k, v in test_batch_inputs.items():
    print(f"  {k:<20}: shape={v.shape}  dtype={v.dtype}")
    if k == 'label_encoded':
        print(f"    min={v.min()}  max={v.max()}  sample={v[0][:10]}")
    elif k == 'label_lengths':
        print(f"    min={v.min()}  max={v.max()}  sample_value={v[0]}")

# Check if any labels contain indices >= 79
labels_batch = test_batch_inputs['label_encoded']
label_lens = test_batch_inputs['label_lengths']

print(f"\nChecking for invalid label indices (>= 79):")
for i in range(min(5, len(labels_batch))):
    label = labels_batch[i][:label_lens[i]]  # only check up to actual length
    if (label >= 79).any():
        print(f"  ⚠️  Sample {i}: contains index >= 79: {label}")
    elif (label < 0).any():
        print(f"  ⚠️  Sample {i}: contains negative index: {label}")
    else:
        max_idx = label.max()
        print(f"  ✓ Sample {i}: max index = {max_idx} (length={label_lens[i]})")

print('\n' + '═' * 70)
print('If you see warnings above, there is a data encoding issue.')
print('Otherwise, proceeding with training...\n')
print('═' * 70)

# Compile model (loss is None — computed inside CTCLayer)
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
)
print('Model compiled.')

# Get callbacks (checkpoint, early stop, reduce LR, CSV logger)
callbacks = get_callbacks(
    checkpoint_dir  = CHECKPOINT_DIR,
    experiment_name = EXPERIMENT,
    patience_stop   = 15,
    patience_lr     = 5
)

print('\n🚀 Starting training...')
print('Watch val_loss: it should decrease over epochs.\n')

history = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = EPOCHS,
    callbacks       = callbacks,
    verbose         = 1
)

print('\n✅ Training done!')


## Step 9 — Plot Training History

**HOW TO READ THE GRAPH:**
- Both curves going DOWN → learning correctly ✅
- Train down, Val UP → overfitting ❌ (add dropout or reduce model size)
- Both flat from start → LR too low ❌ (increase learning rate)
- Loss spikes → LR too high ❌ (reduce learning rate)

In [ ]:
from backend.utils.visualize import plot_training_history
import matplotlib.pyplot as plt

# Plot and save
plot_training_history(
    history.history,
    save_path=f'{CHECKPOINT_DIR}/{EXPERIMENT}_history.png'
)

# Also display inline in Colab
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Val Loss',   color='red')
plt.xlabel('Epoch')
plt.ylabel('CTC Loss')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Best val_loss: {min(history.history["val_loss"]):.4f}')
print(f'Stopped at epoch: {len(history.history["loss"])}')

## Step 10 — Save Models

**WHY TWO MODEL FILES?**
- `_best.weights.h5` = weights only (saved by ModelCheckpoint callback)
- `_inference.keras` = full inference model (architecture + weights together)

The inference model is what you download and use locally for predictions.

In [ ]:
from backend.models.crnn_model import build_inference_model
import os

# Load best weights back into model (in case final epoch wasn't the best)
best_weights_path = f'{CHECKPOINT_DIR}/{EXPERIMENT}_best.weights.h5'
if os.path.exists(best_weights_path):
    model.load_weights(best_weights_path)
    print(f'Best weights loaded from: {best_weights_path}')

# Build inference model (image-only input, no labels needed)
inference_model = build_inference_model(model)

# Save full inference model to Drive
inference_path = f'{CHECKPOINT_DIR}/{EXPERIMENT}_inference.keras'
inference_model.save(inference_path)
print(f'\n✅ Inference model saved → {inference_path}')
print(f'   Download this file and place it in your local saved_models/ folder.')

# Show file size
size_mb = os.path.getsize(inference_path) / (1024 * 1024)
print(f'   File size: {size_mb:.1f} MB')

## Step 11 — Evaluate on Test Set (CER & WER)

**WHY evaluate on TEST set (not val)?**
- Val set was used DURING training (for checkpointing and early stop)
- So val scores are slightly optimistic (model was tuned toward them)
- Test set was NEVER seen during training → unbiased, honest score

**CER (Character Error Rate):**
0.0 = perfect | 0.1 = 10% character errors | 1.0 = completely wrong

State-of-the-art models achieve CER ≈ 0.03–0.05 on IAM.

In [ ]:
from backend.inference.decoder import decode_batch
from backend.utils.metrics     import print_metrics_report, character_error_rate, word_error_rate
import numpy as np

all_predictions  = []
all_ground_truths = []

print('Running evaluation on test set...')
for batch_idx in range(len(test_gen)):
    inputs, _ = test_gen[batch_idx]
    images    = inputs['input_images']

    # Forward pass (inference model — no labels needed)
    preds = inference_model.predict(images, verbose=0)

    # Decode predictions → text
    pred_texts = decode_batch(preds, int_to_char, method='greedy')

    # Get ground truth text from encoded labels
    label_enc = inputs['label_encoded']     # (batch, MAX_LABEL_LEN)
    label_len = inputs['label_lengths']     # (batch,)

    for i in range(len(pred_texts)):
        # Decode ground truth
        indices  = [int(x) for x in label_enc[i][:int(label_len[i])] if x > 0]
        gt_text  = ''.join(int_to_char.get(idx, '?') for idx in indices)
        all_ground_truths.append(gt_text)
        all_predictions.append(pred_texts[i])

# Report
cer = character_error_rate(all_predictions, all_ground_truths)
wer = word_error_rate(all_predictions, all_ground_truths)

print(f'\n=========================================')
print(f'  TEST SET EVALUATION')
print(f'=========================================')
print(f'  Samples evaluated : {len(all_predictions)}')
print(f'  CER               : {cer:.4f} ({cer*100:.2f}%)')
print(f'  WER               : {wer:.4f} ({wer*100:.2f}%)')
print(f'=========================================')

# Show 5 sample predictions
print('\nSample predictions:')
for i in range(min(5, len(all_predictions))):
    gt   = all_ground_truths[i]
    pred = all_predictions[i]
    ok   = '✓' if gt == pred else '✗'
    print(f'  {ok} GT:   "{gt}"')
    print(f'     Pred: "{pred}"')
    print()

## Step 12 — Run Inference on a New Image

**WHY this is the final goal:**
Everything we built leads to this — point the model at any handwritten
image and get the text out, without any external API.

Upload a handwritten image to your Drive `sample_images/` folder and change the path below.

In [ ]:
import matplotlib.pyplot as plt
import cv2
from backend.preprocessing.image_processor import preprocess_image
from backend.inference.decoder             import decode_batch
import numpy as np
import pandas as pd

# ✏️ To test your own image: upload it to Drive and set the path here
TEST_IMAGE_PATH = 'sample_images/test.png'

if not os.path.exists(TEST_IMAGE_PATH):
    # Fall back to a random sample from the test set
    test_df_local = pd.read_csv('data/iam_words/splits/test.csv')
    row = test_df_local.sample(1, random_state=7).iloc[0]
    TEST_IMAGE_PATH = row['image_path']
    print(f'Using test sample: {os.path.basename(TEST_IMAGE_PATH)}')
    print(f'Ground truth: "{row["label"]}"')

# Preprocess → model expects (1, 64, 256, 1)
img       = preprocess_image(TEST_IMAGE_PATH)
img_batch = np.expand_dims(img, axis=0)

# Predict
preds     = inference_model.predict(img_batch, verbose=0)
pred_text = decode_batch(preds, int_to_char, method='greedy')[0]

# Display side by side
original = cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Input Image (original)')
axes[0].axis('off')
axes[1].imshow(img[:,:,0], cmap='gray')
axes[1].set_title(f'Preprocessed 64×256 | Predicted: "{pred_text}"')
axes[1].axis('off')
plt.suptitle('OCR Inference Result', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n🎯 Extracted Text: "{pred_text}"')


## Step 13 — Download the Trained Model

After training, download the inference model to your local machine.
Then place it in `d:\text\ocr_project\saved_models\` and use `predict.py` locally.

In [ ]:
import os

inference_path = f'{CHECKPOINT_DIR}/{EXPERIMENT}_inference.keras'

# Verify the file exists
if os.path.exists(inference_path):
    size_mb = os.path.getsize(inference_path) / (1024 * 1024)
    print(f'✅ Inference model saved: {inference_path}')
    print(f'   File size: {size_mb:.1f} MB')
    print('\n📥 TO DOWNLOAD FROM KAGGLE:')
    print('   1. Click "Output" tab at the top of this notebook')
    print('   2. Find crnn_iam_v1_inference.keras in the file list')
    print('   3. Click the ⋮ (three dots) → Download')
    print('   4. Place it in: d:\\text\\ocr_project\\saved_models\\')
    print('\n   Then run locally:')
    print('   python backend/inference/predict.py --image_path sample_images/test.png --model_path saved_models/crnn_iam_v1_inference.keras')
else:
    print(f'❌ Model file not found: {inference_path}')